# Actividad 5 – Análisis de ventas por tienda y producto
### Iván López - A01284875

## Metodología
* Leer el archivo csv con PySpark.
* Construir columnas derivadas (fecha, revenue).
* Calcular KPIs diarios por tienda y top productos por canal.
* Exportar resultados a formatos analíticos (Parquet).

## Librerías requeridas

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

## Configuración de Spark

In [2]:
# Crear sesión Spark
try:
    sc.stop()
except Exception:
    pass

In [3]:
spark = (SparkSession.builder
         .appName("NotebookSession")
         .master("local[*]")
         .config("spark.ui.port", "0")
         .getOrCreate())

sc = spark.sparkContext
sc.setLogLevel("WARN")

print(spark.version, sc.appName)

4.0.1 NotebookSession


## Carga de datos

In [4]:
# leer el CSV
df_sales = spark.read.csv('activity5_sales_big.csv', header=True, inferSchema=True) 

In [5]:
# verificar esquema
df_sales.printSchema()

root
 |-- sale_id: string (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- sale_ts: timestamp (nullable = true)
 |-- units: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- channel: string (nullable = true)



In [6]:
#primeras filas
df_sales.show(10)

+--------------------+--------+----------+-------------------+-----+----------+-------+
|             sale_id|store_id|product_id|            sale_ts|units|unit_price|channel|
+--------------------+--------+----------+-------------------+-----+----------+-------+
|41a096ae-3ce8-454...|       3|       421|2025-01-29 15:23:00|    2|     695.6| ONLINE|
|cd083bb8-852f-4b5...|      50|       362|2025-01-04 18:17:00|    2|     671.5|  STORE|
|ed7de8eb-4b42-416...|      34|       153|2025-01-28 17:31:00|    2|    1026.2| ONLINE|
|966e7922-ef9c-4dc...|      49|        32|2025-03-18 18:20:00|    5|    414.53|    APP|
|f5356e57-2a5e-4df...|       7|       475|2025-03-18 02:12:00|    2|    972.19|    APP|
|5d94db10-b085-4ac...|       6|        59|2025-01-09 15:27:00|    5|    472.45|    APP|
|ec84acd2-b726-440...|      40|       153|2025-02-09 21:24:00|    2|    917.64| ONLINE|
|c461b5ca-084f-431...|      37|       392|2025-03-27 06:23:00|    3|    742.15|  STORE|
|a3ef1969-7362-416...|       3| 

## Columnas derivadas

In [ ]:
#Convertir sale_ts a timestamp. (ya está)
#Derivar sale_date usando to_date.
#Crear la columna revenue = units * unit_price.

df_sales = (df_sales
            .withColumn('sale_ts', F.to_timestamp('sale_ts'))
            .withColumn('sale_date', F.to_date('sale_ts'))
            .withColumn('revenue', F.col('units')*F.col('unit_price'))) 

df_sales.show(10)

+--------------------+--------+----------+-------------------+-----+----------+-------+----------+------------------+
|             sale_id|store_id|product_id|            sale_ts|units|unit_price|channel| sale_date|           revenue|
+--------------------+--------+----------+-------------------+-----+----------+-------+----------+------------------+
|41a096ae-3ce8-454...|       3|       421|2025-01-29 15:23:00|    2|     695.6| ONLINE|2025-01-29|            1391.2|
|cd083bb8-852f-4b5...|      50|       362|2025-01-04 18:17:00|    2|     671.5|  STORE|2025-01-04|            1343.0|
|ed7de8eb-4b42-416...|      34|       153|2025-01-28 17:31:00|    2|    1026.2| ONLINE|2025-01-28|            2052.4|
|966e7922-ef9c-4dc...|      49|        32|2025-03-18 18:20:00|    5|    414.53|    APP|2025-03-18|2072.6499999999996|
|f5356e57-2a5e-4df...|       7|       475|2025-03-18 02:12:00|    2|    972.19|    APP|2025-03-18|           1944.38|
|5d94db10-b085-4ac...|       6|        59|2025-01-09 15:

## KPIs diarios por tienda

In [11]:
daily_kpis=(df_sales.groupBy('sale_date','store_id')
            .agg(F.round(F.sum('revenue'),2).alias('total_revenue'),
            F.sum('units').alias('total_units'))
            .orderBy(F.desc('total_revenue')))

daily_kpis.show(10)

+----------+--------+-------------+-----------+
| sale_date|store_id|total_revenue|total_units|
+----------+--------+-------------+-----------+
|2025-01-18|      36|     49466.39|         77|
|2025-01-29|      19|     48784.56|         62|
|2025-03-16|      14|     48653.11|         66|
|2025-03-07|      14|     48572.64|         69|
|2025-03-27|      35|     48147.54|         66|
|2025-03-29|      17|     46169.47|         55|
|2025-02-12|      48|     46068.11|         64|
|2025-03-14|      30|     43542.11|         50|
|2025-03-11|       1|     41887.78|         58|
|2025-02-02|      23|     41820.53|         50|
+----------+--------+-------------+-----------+
only showing top 10 rows


## Top productos

In [12]:
top_products=(df_sales.groupBy('channel','product_id')
            .agg(F.round(F.sum('revenue'),2).alias('total_revenue'),
            F.sum('units').alias('total_units'))
            .orderBy(F.desc('total_revenue')))

top_products.show(10)

+-------+----------+-------------+-----------+
|channel|product_id|total_revenue|total_units|
+-------+----------+-------------+-----------+
|    APP|       108|      96684.9|        161|
|  STORE|       319|     90298.49|        132|
|    APP|        36|     88682.21|        140|
|  STORE|       339|     85301.11|        117|
| ONLINE|       222|     84762.15|        101|
| ONLINE|       219|     84555.38|        126|
|  STORE|       311|     83015.46|        132|
|  STORE|       444|      82612.8|        109|
| ONLINE|        76|     82181.14|        115|
| ONLINE|        96|     81537.64|        131|
+-------+----------+-------------+-----------+
only showing top 10 rows


## Exportar resultados

In [15]:
daily_kpis.write.mode("overwrite").parquet("../actividad5/outputs/activity5_daily_store.parquet")
top_products.write.mode("overwrite").parquet("../actividad5/outputs/activity5_top_products.parquet")

print('Resultados exportados exitosamente a Parquet.')

Resultados exportados exitosamente a Parquet.
